# ERA5-Land MIN3P Zarr — QC notebook
Reads the output zarr from `era5_land_edh_min3p.py` and runs a handful of
sanity checks: structure, coverage, missing values, and per-variable time series.

In [ ]:
import s3fs
import xarray as xr
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

In [ ]:
# ── change this to match the zarr you created ──────────────────────────────
ZARR_PATH = (
    "s3://carbonplan-carbon-removal/ew-workflows-data/min3p/era5-land/"
    "era5-land_2000-01-01_2020-12-31_hourly.zarr"
)
# ───────────────────────────────────────────────────────────────────────────

## 1. Open and inspect

In [ ]:
fs = s3fs.S3FileSystem(anon=False)
ds = xr.open_zarr(ZARR_PATH, storage_options={"anon": False})
ds

In [ ]:
print("=== Dataset attributes ===")
for k, v in ds.attrs.items():
    print(f"  {k}: {v}")

print("\n=== Variable attributes ===")
for var in ds.data_vars:
    print(f"  {var}: {ds[var].attrs}")

## 2. Time coverage

In [ ]:
t = ds.valid_time.values
print(f"First timestamp : {t[0]}")
print(f"Last  timestamp : {t[-1]}")
print(f"N timesteps     : {len(t)}")

# Check for gaps — consecutive differences should all be exactly 1 hour
dt = np.diff(t).astype("timedelta64[h]").astype(int)
gaps = np.where(dt != 1)[0]
if len(gaps) == 0:
    print("No time gaps detected. ✓")
else:
    print(f"WARNING: {len(gaps)} gaps found at indices: {gaps[:10]}")
    print("  Gap sizes (h):", dt[gaps[:10]])

## 3. Sites

In [ ]:
site_df = pd.DataFrame({
    "site_id"   : ds.site_id.values,
    "target_lat": ds.target_lat.values,
    "target_lon": ds.target_lon.values,
    "snapped_lat": ds.latitude.values,
    "snapped_lon": (ds.longitude.values + 180) % 360 - 180,  # 0–360 → ±180
})
site_df["lat_offset"] = (site_df.snapped_lat - site_df.target_lat).round(4)
site_df["lon_offset"] = (site_df.snapped_lon - site_df.target_lon).round(4)
display(site_df)

## 4. Missing values

In [ ]:
n_sites = ds.dims["site"]
print("NaN counts per variable:")
for var in ds.data_vars:
    n = int(ds[var].isnull().sum().compute())
    is_accum = "deaccumulated" in ds[var].attrs.get("processing", "")
    expected = n_sites if is_accum else 0
    if n == expected:
        flag = " ✓ (1 NaN per site at first timestamp, expected)" if is_accum else " ✓"
    else:
        flag = f" ← check (expected {expected})"
    print(f"  {var:6s}: {n}{flag}")

## 5. Basic statistics

In [ ]:
rows = []
for var in ds.data_vars:
    da = ds[var].compute()
    rows.append({
        "variable": var,
        "units"   : ds[var].attrs.get("units", ""),
        "min"     : float(da.min()),
        "mean"    : float(da.mean()),
        "max"     : float(da.max()),
    })

stats = pd.DataFrame(rows).set_index("variable")
display(stats.round(3))

**Expected ranges (rough):**
- `t2m` / `d2m`: 220–330 K
- `u10` / `v10`: roughly –30 to +30 m s⁻¹
- `ssr`: ≥ 0 W m⁻² (solar radiation, should not be negative)
- `str`: can be negative (net thermal can go either direction)
- `tp`: ≥ 0 mm day⁻¹ (precipitation cannot be negative)
- `sp`: 50,000–110,000 Pa (sea-level ≈ 101,325 Pa; lower for elevated sites)

In [ ]:
# Flag physically implausible values
checks = {
    "t2m" : lambda da: (da < 180) | (da > 340),
    "d2m" : lambda da: (da < 180) | (da > 340),
    "u10" : lambda da: abs(da) > 100,
    "v10" : lambda da: abs(da) > 100,
    "ssr" : lambda da: da < -1,   # small negatives can be rounding artifacts
    "tp"  : lambda da: da < -0.1,  # deaccum can produce tiny negatives; flag large ones
    "sp"  : lambda da: (da < 50_000) | (da > 110_000),
}
for var, test in checks.items():
    if var not in ds:
        continue
    n = int(test(ds[var]).sum().compute())
    flag = " ← PROBLEM" if n > 0 else " ✓"
    print(f"  {var}: {n} implausible values{flag}")

## 6. Time series — all variables, first 90 days

In [ ]:
# Use first site for the overview time series
SITE_IDX = 0
site_label = str(ds.site_id.values[SITE_IDX])

ds_site = ds.isel(site=SITE_IDX).sel(valid_time=slice(None, None)).compute()
t90     = ds_site.valid_time.values[:24 * 90]  # first 90 days

fig, axes = plt.subplots(len(ds.data_vars), 1, figsize=(14, 2.5 * len(ds.data_vars)),
                         sharex=True)
for ax, var in zip(axes, ds.data_vars):
    vals = ds_site[var].values[:24 * 90]
    ax.plot(t90, vals, lw=0.6)
    ax.set_ylabel(f"{var}\n({ds[var].attrs.get('units', '?')})", fontsize=9)
    ax.grid(True, alpha=0.3)

axes[-1].xaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))
fig.suptitle(f"First 90 days — site {site_label}", fontsize=11)
fig.autofmt_xdate()
plt.tight_layout()
plt.show()

## 7. Radiation: verify deaccumulation looks right
Plot a single day of `ssr` at hourly resolution — should be zero at night and
peak around solar noon, with no large jumps at midnight.

In [ ]:
# Pick a summer day (2000-07-15) for a clear solar signal
day_slice = slice("2000-07-15", "2000-07-16")
ssr_day   = ds_site["ssr"].sel(valid_time=day_slice).compute()
str_day   = ds_site["str"].sel(valid_time=day_slice).compute()

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 5), sharex=True)
ax1.bar(ssr_day.valid_time.values, ssr_day.values, width=1/24, align="edge", color="orange")
ax1.set_ylabel("ssr (W m⁻²)")
ax1.set_title(f"Surface net solar radiation — site {site_label}")
ax1.grid(True, alpha=0.3)

ax2.bar(str_day.valid_time.values, str_day.values, width=1/24, align="edge", color="steelblue")
ax2.set_ylabel("str (W m⁻²)")
ax2.set_title("Surface net thermal radiation")
ax2.grid(True, alpha=0.3)

ax2.xaxis.set_major_formatter(mdates.DateFormatter("%H:%M"))
fig.autofmt_xdate()
plt.tight_layout()
plt.show()

## 8. Seasonal means — all sites

In [ ]:
ds_monthly = ds.resample(valid_time="1ME").mean().compute()

n_sites = ds.dims["site"]
fig, axes = plt.subplots(len(ds.data_vars), 1, figsize=(14, 2.5 * len(ds.data_vars)),
                         sharex=True)
colors = plt.cm.tab10.colors

for ax, var in zip(axes, ds.data_vars):
    for i in range(n_sites):
        label = str(ds.site_id.values[i])
        ax.plot(
            ds_monthly.valid_time.values,
            ds_monthly[var].isel(site=i).values,
            lw=1.2, color=colors[i % 10], label=label,
        )
    ax.set_ylabel(f"{var}\n({ds[var].attrs.get('units', '?')})", fontsize=9)
    ax.grid(True, alpha=0.3)

axes[0].legend(fontsize=7, ncol=min(n_sites, 5), loc="upper right")
axes[-1].xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
fig.suptitle("Monthly means — all sites", fontsize=11)
fig.autofmt_xdate()
plt.tight_layout()
plt.show()

## 9. Wind speed sanity check

In [ ]:
ws = np.sqrt(ds["u10"] ** 2 + ds["v10"] ** 2).compute()

fig, axes = plt.subplots(1, n_sites, figsize=(4 * n_sites, 3), sharey=True)
if n_sites == 1:
    axes = [axes]

for i, ax in enumerate(axes):
    ax.hist(ws.isel(site=i).values.ravel(), bins=60, density=True, color=colors[i % 10])
    ax.set_title(str(ds.site_id.values[i]), fontsize=9)
    ax.set_xlabel("wind speed (m s⁻¹)")

axes[0].set_ylabel("density")
fig.suptitle("Hourly wind speed distribution", fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:
# ---